In [4]:

import os
import numpy as np
import pandas as pd
import librosa
from sklearn.preprocessing import StandardScaler

In [5]:
# Read each .wav file and extract MFCC or Spectogram features
def extract_features(file_path, feature_type='mfcc', n_mfcc=13, n_fft=2048, hop_length=512):
    try:
        y, sr = librosa.load(file_path, sr=None)
        if feature_type.lower() == 'mfcc':
            features = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc, n_fft=n_fft, hop_length=hop_length)
        elif feature_type.lower() == 'spectrogram':
            stft = np.abs(librosa.stft(y, n_fft=n_fft, hop_length=hop_length))
            features = librosa.power_to_db(stft**2)
        else:
            raise ValueError("Invalid feature_type: choose 'mfcc' or 'spectrogram'")
        return features.T
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None

In [7]:
# Scales and equalizes all feature matrices
def normalize_and_pad(features_list, max_len=None):
    if not max_len:
        max_len = max(f.shape[0] for f in features_list if f is not None)

    scaler = StandardScaler()
    padded = []
    for f in features_list:
        if f is None: continue
        f = scaler.fit_transform(f)
        if f.shape[0] < max_len:
            pad = max_len - f.shape[0]
            f = np.pad(f, ((0,pad),(0,0)), mode='constant')
        else:
            f = f[:max_len, :]
        padded.append(f)
    return np.array(padded)

In [8]:
# Reads CSV files and saves the results (.npy files) in results folder
def preprocess_dataset(csv_path, feature_type='mfcc', save_dir='results', is_train=True):
    df = pd.read_csv(csv_path)
    features_list, labels = [], []

    print(f"\nExtracting {feature_type.upper()} features from {len(df)} files...")

    for _, row in df.iterrows():
        fpath = row['file_path']
        feat = extract_features(fpath, feature_type)
        if feat is not None:
            features_list.append(feat)
            if is_train:
                labels.append(row.get('class', None))

    print("Feature extraction done. Normalizing and padding...")

    X = normalize_and_pad(features_list)
    os.makedirs(save_dir, exist_ok=True)

    if is_train:
        y = np.array(labels, dtype=np.float32)
        np.save(os.path.join(save_dir, 'X_train.npy'), X)
        np.save(os.path.join(save_dir, 'y_train.npy'), y)
        print(f"Saved X_train.npy and y_train.npy in {save_dir}")
        return X, y
    else:
        np.save(os.path.join(save_dir, 'X_test.npy'), X)
        print(f"Saved X_test.npy in {save_dir}")
        return X, None

In [9]:
if __name__ == "__main__":
    base_dir = os.path.dirname(__file__) if "__file__" in globals() else os.getcwd()
    out_dir = os.path.join(base_dir, "results")

    train_csv = os.path.join(out_dir, "train_data.csv")
    test_csv = os.path.join(out_dir, "test_data.csv")

    preprocess_dataset(train_csv, 'mfcc', out_dir, True)
    preprocess_dataset(test_csv, 'mfcc', out_dir, False)


Extracting MFCC features from 2176 files...
Feature extraction done. Normalizing and padding...
Saved X_train.npy and y_train.npy in d:\University\AI\SAND Project\Detecting_Dysarthia\Baseline Pipeline Implementation\results

Extracting MFCC features from 784 files...
Feature extraction done. Normalizing and padding...
Saved X_test.npy in d:\University\AI\SAND Project\Detecting_Dysarthia\Baseline Pipeline Implementation\results
